# Chapter 12 &mdash; Exploring PDA in Jove: Reading ID Traces

**Concept 6 of the Chapter 12 decomposition:** *Exploring PDA in Jove: Reading ID Traces*

`explore_pda(s, P)` prints every accepting computation as a chain of IDs.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter12/Concept-Exploring-PDA-In-Jove/Concept-Exploring-PDA-In-Jove.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.Def_PDA        import *
from jove.AnimatePDA     import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


`explore_pda(s, P)` is the debugging tool for PDA. It prints, for the given input:

* whether the string is **accepted**;
* **every** accepting computation, as a chain of IDs;
* the surviving and visited IDs, so you can see where runs died.

Reading a trace teaches more than reading the transition table: you see the stack
grow, the nondeterministic branches fork, and the dead ends die.

`run_pda` returns the same information as data &mdash; a triple *(surviving, accepting
paths, visited)* &mdash; which is what you assert on in tests.

## 2. Definitions

### A machine with genuine nondeterminism

# --- a thin wrapper over Jove's PDA runner -----------------------------
# run_pda returns (surviving-IDs, accepting-paths, visited-IDs); a string
# is accepted exactly when the list of accepting paths is non-empty.
def pda_accepts(P, s, acceptance='ACCEPT_F', STKMAX=6):
    surv, paths, visited = run_pda(s, P, acceptance=acceptance, STKMAX=STKMAX)
    return len(paths) > 0

def pda_npaths(P, s, acceptance='ACCEPT_F', STKMAX=6):
    return len(run_pda(s, P, acceptance=acceptance, STKMAX=STKMAX)[1])

In [ ]:
AbOrAc = md2mc('''PDA
!! { a^i b^j c^k : i = j or i = k }
I  : a , # ; A#  -> I
I  : a , A ; AA  -> I
I  : '' , # ; #  -> B     !! guess: match the b's
I  : '' , A ; A  -> B
I  : '' , # ; #  -> C     !! guess: match the c's
I  : '' , A ; A  -> C
B  : b , A ; ''  -> B
B  : '' , # ; #  -> BC
BC : c , # ; #   -> BC
BC : '' , # ; #  -> F
C  : b , A ; A   -> C     !! skip the b's, keep the count
C  : b , # ; #   -> C     !! ... and skip them when no a's were counted at all
C  : '' , A ; A  -> CC
C  : '' , # ; #  -> CC
CC : c , A ; ''  -> CC
CC : '' , # ; #  -> F
''')

# --- a thin wrapper over Jove's PDA runner -----------------------------
# run_pda returns (surviving-IDs, accepting-paths, visited-IDs); a string
# is accepted exactly when the list of accepting paths is non-empty.
def pda_accepts(P, s, acceptance='ACCEPT_F', STKMAX=6):
    surv, paths, visited = run_pda(s, P, acceptance=acceptance, STKMAX=STKMAX)
    return len(paths) > 0

def pda_npaths(P, s, acceptance='ACCEPT_F', STKMAX=6):
    return len(run_pda(s, P, acceptance=acceptance, STKMAX=STKMAX)[1])

### The specification

In [ ]:
def in_abac(s):
    i = len(s) - len(s.lstrip('a')); rest = s[i:]
    j = len(rest) - len(rest.lstrip('b')); k = len(rest) - j
    return s == 'a'*i + 'b'*j + 'c'*k and (i == j or i == k)

## 3. Tests

`explore_pda` prints the whole story.

In [ ]:
explore_pda('aabbc', AbOrAc, STKMAX=8)

`run_pda` gives the same thing as data.

In [ ]:
surv, paths, visited = run_pda('aabb', AbOrAc, STKMAX=8)
print("accepting computations :", len(paths))
print("visited IDs            :", len(visited))
for idd in paths[0][1][:6]:
    print("   ", idd)
assert paths

Nondeterminism shows up as **several** accepting computations.

In [ ]:
for s in ['aabbcc', 'aabb', 'aacc', 'abc']:
    n = pda_npaths(AbOrAc, s, STKMAX=8)
    print("  %-8r accepting computations : %d" % (s, n))
assert pda_npaths(AbOrAc, 'abc', STKMAX=8) >= 2
print("\n'abc' has i=j AND i=k, so both guesses succeed.")

Dead ends are visible in the visited set but contribute no path.

In [ ]:
surv, paths, visited = run_pda('aabc', AbOrAc, STKMAX=8)   # i=2, j=1, k=1
print("accepted? ", bool(paths), "  IDs explored :", len(visited))
assert not paths
print("\nthe machine explored %d IDs and every branch died." % len(visited))
print("(note 'abbc' has i=k=1, so it IS in L -- read the disjunction carefully)")
assert pda_accepts(AbOrAc, 'abbc', STKMAX=8)

And the machine is correct.

In [ ]:
from itertools import product
strs = [''.join(p) for k in range(5) for p in product('abc', repeat=k)]
bad = [s for s in strs if pda_accepts(AbOrAc, s, STKMAX=6) != in_abac(s)]
print("mismatches over %d strings :" % len(strs), bad)
assert not bad

## 4. Animation

The nondeterministic machine; the two guesses are the two $\varepsilon$ branches out of `I`.

*(The `display(HTML(...))` line loads the toolbar's font-awesome icons. Keep it last in the cell &mdash; it must be there for the controls to appear.)*

In [ ]:
from jove.AnimatePDA import *
AnimatePDA(AbOrAc, FuseEdges=True)
display(HTML('<link rel="stylesheet" href="//stackpath.bootstrapcdn.com/font-awesome/4.7.0/css/font-awesome.min.css"/>'))

## 5. Exercises


1. Run `explore_pda` on a rejected string. What does it show?
2. Which ID in the trace is the moment the guess is committed?
3. Add `chatty=True` to `run_pda`. What extra detail appears?

In [ ]:
# Your work for the exercises above.